# T2ICount Colab bootstrap
This notebook only orchestrates repository setup, validation, and training commands. Model, loss, and inference logic remain in repository modules.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# -------------------------
# Persistent files on Drive
# -------------------------
DRIVE_ZIP = Path("/content/drive/MyDrive/T2ICount-assets.zip")

# Keep training checkpoints persistent on Drive
DRIVE_ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"
DRIVE_CHECKPOINT_ROOT = (
    Path(DRIVE_ASSET_ROOT) / "checkpoints/baseline_retrain"
)
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------
# Fast local Colab storage
# -------------------------
LOCAL_ZIP = Path("/content/T2ICount-assets.zip")

# IMPORTANT:
# ASSET_ROOT now points to Colab local SSD, NOT Google Drive.
ASSET_ROOT = "/content/T2ICount-assets"
ASSET_ROOT_PATH = Path(ASSET_ROOT)

REPO_DIR = "/content/T2ICount-Implementation"

# Infrastructure smoke tests are already validated. Keep this False for
# normal experiments; enable only after infrastructure changes.
RUN_INFRA_SMOKE_TESTS = False

print("Drive ZIP :", DRIVE_ZIP)
print("Local assets:", ASSET_ROOT)
print("Drive saves :", DRIVE_ASSET_ROOT)

Mounted at /content/drive
Drive ZIP : /content/drive/MyDrive/T2ICount-assets.zip
Local assets: /content/T2ICount-assets
Drive saves : /content/drive/MyDrive/T2ICount-assets


In [2]:
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

from tqdm.auto import tqdm


# ============================================================
# Required assets
# ============================================================

REQUIRED_ASSETS = {
    "CLIP": (
        ASSET_ROOT_PATH
        / "pretrained/clip-vit-large-patch14",
        "directory",
    ),

    "Stable Diffusion": (
        ASSET_ROOT_PATH
        / "pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt",
        "file",
    ),

    "FSC147": (
        ASSET_ROOT_PATH
        / "datasets/FSC147",
        "directory",
    ),

    "T2ICount checkpoint": (
        ASSET_ROOT_PATH
        / "checkpoints/official/best_model_paper.pth",
        "file",
    ),
}


def check_assets(verbose=True):
    status = {}
    for name, (path, kind) in REQUIRED_ASSETS.items():
        status[name] = path.is_dir() if kind == "directory" else path.is_file()

    if verbose:
        print("\nAsset status:")

        for name, ok in status.items():
            tag = "OK" if ok else "MISSING"

            print(f"  [{tag}] {name}")

            if not ok:
                print(f"       {REQUIRED_ASSETS[name][0]}")

    return all(status.values())


def human_size(num_bytes):
    units = ["B", "KB", "MB", "GB", "TB"]

    size = float(num_bytes)

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"

        size /= 1024

    return f"{size:.2f} PB"


def copy_with_progress(src, dst, chunk_size=64 * 1024 * 1024):
    """
    Copy one large file from Drive -> local SSD with progress.
    """

    total = src.stat().st_size

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with src.open("rb") as fsrc, dst.open("wb") as fdst:
        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Copy ZIP Drive -> local",
        ) as progress:

            while True:
                chunk = fsrc.read(chunk_size)

                if not chunk:
                    break

                fdst.write(chunk)
                progress.update(len(chunk))


# ============================================================
# 1. Skip everything if local assets are already complete
# ============================================================

if check_assets():
    print(
        "\nLocal assets already complete:",
        ASSET_ROOT_PATH,
    )

else:

    # ========================================================
    # 2. ZIP must exist on Drive
    # ========================================================

    if not DRIVE_ZIP.is_file():
        raise FileNotFoundError(
            "\nCannot find asset ZIP on Google Drive:\n"
            f"{DRIVE_ZIP}\n\n"
            "Upload T2ICount-assets.zip to My Drive first."
        )

    drive_zip_size = DRIVE_ZIP.stat().st_size

    print("\nDrive ZIP found.")
    print("ZIP size:", human_size(drive_zip_size))


    # ========================================================
    # 3. Reuse local ZIP if it was already copied successfully
    # ========================================================

    local_zip_valid = (
        LOCAL_ZIP.is_file()
        and LOCAL_ZIP.stat().st_size == drive_zip_size
    )

    if local_zip_valid:
        print(
            "\nLocal ZIP already exists with matching size."
        )
        print("Skipping Drive copy:", LOCAL_ZIP)

    else:

        # Remove ONLY an incomplete LOCAL ZIP.
        # Never deletes anything from Google Drive.
        if LOCAL_ZIP.exists():
            print(
                "\nRemoving incomplete local ZIP:",
                LOCAL_ZIP,
            )
            LOCAL_ZIP.unlink()

        free_space = shutil.disk_usage("/content").free

        print(
            "Free local disk:",
            human_size(free_space),
        )

        if free_space < drive_zip_size + 5 * 1024**3:
            raise RuntimeError(
                "Not enough local disk space to safely copy "
                "the asset archive."
            )

        print(
            "\nCopying ZIP from Google Drive to Colab SSD..."
        )

        start = time.perf_counter()

        copy_with_progress(
            DRIVE_ZIP,
            LOCAL_ZIP,
        )

        elapsed = time.perf_counter() - start

        print(
            f"\nZIP copied in {elapsed / 60:.1f} minutes."
        )


    # ========================================================
    # 4. Validate ZIP
    # ========================================================

    print("\nChecking ZIP integrity...")

    if not zipfile.is_zipfile(LOCAL_ZIP):
        raise RuntimeError(
            f"Local archive is not a valid ZIP:\n{LOCAL_ZIP}"
        )


    # ========================================================
    # 5. Determine archive layout
    # ========================================================

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:

        members = zf.infolist()

        names = [
            info.filename.replace("\\", "/")
            for info in members
        ]

        has_root_folder = any(
            name.startswith("T2ICount-assets/")
            for name in names
        )

        total_uncompressed = sum(
            info.file_size
            for info in members
        )

    print(
        "Uncompressed size:",
        human_size(total_uncompressed),
    )

    print(
        "Files in archive:",
        len(members),
    )


    if has_root_folder:

        # ZIP structure:
        #
        # T2ICount-assets/
        # ├── pretrained/
        # ├── datasets/
        # └── checkpoints/
        #
        # Extract to /content

        extract_to = Path("/content")

    else:

        # ZIP structure:
        #
        # pretrained/
        # datasets/
        # checkpoints/
        #
        # Extract directly to asset root

        ASSET_ROOT_PATH.mkdir(
            parents=True,
            exist_ok=True,
        )

        extract_to = ASSET_ROOT_PATH


    print(
        "\nArchive layout:",
        (
            "contains T2ICount-assets/"
            if has_root_folder
            else "contents start inside asset root"
        ),
    )

    print("Extract target:", extract_to)


    # ========================================================
    # 6. Extract on LOCAL SSD
    # ========================================================

    print(
        "\nExtracting on Colab local SSD..."
    )
    print(
        "This should be much faster than extracting "
        "directly into Google Drive."
    )

    start = time.perf_counter()

    # Use system unzip: significantly better suited for
    # extracting many files locally than doing Drive FUSE I/O.
    result = subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(LOCAL_ZIP),
            "-d",
            str(extract_to),
        ],
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"unzip failed with return code "
            f"{result.returncode}"
        )

    elapsed = time.perf_counter() - start

    print(
        f"Extraction finished in "
        f"{elapsed / 60:.1f} minutes."
    )


    # ========================================================
    # 7. Validate extracted assets
    # ========================================================

    print("\nValidating extracted assets...")

    if not check_assets():
        raise RuntimeError(
            "\nExtraction finished but required assets "
            "are still missing.\n"
            "Do NOT continue to model validation/training."
        )


    # ========================================================
    # 8. Free local space
    # ========================================================

    # ZIP is no longer needed after successful extraction.
    if LOCAL_ZIP.exists():
        print(
            "\nRemoving local ZIP to recover disk space..."
        )
        LOCAL_ZIP.unlink()


    print("\n========================================")
    print("LOCAL ASSET BOOTSTRAP PASSED")
    print("========================================")
    print("Runtime assets :", ASSET_ROOT_PATH)
    print("Checkpoints    :", DRIVE_CHECKPOINT_ROOT)


Asset status:
  [MISSING] CLIP
       /content/T2ICount-assets/pretrained/clip-vit-large-patch14
  [MISSING] Stable Diffusion
       /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
  [MISSING] FSC147
       /content/T2ICount-assets/datasets/FSC147
  [MISSING] T2ICount checkpoint
       /content/T2ICount-assets/checkpoints/official/best_model_paper.pth

Drive ZIP found.
ZIP size: 12.96 GB
Free local disk: 65.56 GB

Copying ZIP from Google Drive to Colab SSD...


Copy ZIP Drive -> local:   0%|          | 0.00/13.0G [00:00<?, ?B/s]


ZIP copied in 2.9 minutes.

Checking ZIP integrity...
Uncompressed size: 17.86 GB
Files in archive: 15781

Archive layout: contains T2ICount-assets/
Extract target: /content

Extracting on Colab local SSD...
This should be much faster than extracting directly into Google Drive.
Extraction finished in 2.9 minutes.

Validating extracted assets...

Asset status:
  [OK] CLIP
  [OK] Stable Diffusion
  [OK] FSC147
  [OK] T2ICount checkpoint

Removing local ZIP to recover disk space...

LOCAL ASSET BOOTSTRAP PASSED
Runtime assets : /content/T2ICount-assets
Checkpoints    : /content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain


## 2. Clone or update the repository

In [3]:
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/bqa100507-spec/T2ICount-Implementation.git"
repo_path = pathlib.Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
elif not (repo_path / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("Repository:", pathlib.Path.cwd())

Repository: /content/T2ICount-Implementation


In [4]:
%cd /content/T2ICount-Implementation
!git pull --ff-only

/content
Already up to date.


## 3. Install Colab dependencies
If installation replaces important runtime packages such as PyTorch, Transformers, or Lightning, restart the runtime/kernel before continuing when required. Do not continue validation with a partially reloaded environment.

In [5]:
%pip install -r requirements-colab.txt

Obtaining taming-transformers from git+https://github.com/CompVis/taming-transformers.git@3ba01b241669f5ade541ce990f7650a3b8f65318#egg=taming-transformers (from -r requirements-colab.txt (line 14))
  Cloning https://github.com/CompVis/taming-transformers.git (to revision 3ba01b241669f5ade541ce990f7650a3b8f65318) to ./src/taming-transformers
  Running command git clone --filter=blob:none --quiet https://github.com/CompVis/taming-transformers.git /content/T2ICount-Implementation/src/taming-transformers
  Running command git rev-parse -q --verify 'sha^3ba01b241669f5ade541ce990f7650a3b8f65318'
  Running command git fetch -q https://github.com/CompVis/taming-transformers.git 3ba01b241669f5ade541ce990f7650a3b8f65318
  Resolved https://github.com/CompVis/taming-transformers.git to commit 3ba01b241669f5ade541ce990f7650a3b8f65318
  Preparing metadata (setup.py) ... done
Obtaining clip from git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6#egg=clip (from -r requirem

## 4. Configure the notebook Python environment

In [ ]:
import os

os.environ["T2ICOUNT_ASSET_ROOT"] = ASSET_ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("T2ICOUNT_ASSET_ROOT=", os.environ["T2ICOUNT_ASSET_ROOT"] )

T2ICOUNT_ASSET_ROOT= /content/T2ICount-assets


## 5. Validate local runtime assets and offline CLIP loading

In [7]:
from pathlib import Path
import os

ASSET_ROOT = "/content/T2ICount-assets"
ASSET_ROOT_PATH = Path(ASSET_ROOT)

DRIVE_ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"

os.environ["T2ICOUNT_ASSET_ROOT"] = ASSET_ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("ASSET_ROOT =", ASSET_ROOT)
print("DRIVE_ASSET_ROOT =", DRIVE_ASSET_ROOT)
print("T2ICOUNT_ASSET_ROOT =", os.environ["T2ICOUNT_ASSET_ROOT"])

if not check_assets(verbose=True):
    raise FileNotFoundError(
        "Required runtime assets are incomplete under ASSET_ROOT."
    )

ASSET_ROOT = /content/T2ICount-assets
DRIVE_ASSET_ROOT = /content/drive/MyDrive/T2ICount-assets
T2ICOUNT_ASSET_ROOT = /content/T2ICount-assets

Asset status:
  [OK] CLIP
  [OK] Stable Diffusion
  [OK] FSC147
  [OK] T2ICount checkpoint


In [8]:
import subprocess
import sys

subprocess.run([
    sys.executable, "scripts/check_assets.py",
    "--asset-root", ASSET_ROOT,
    "--data", "fsc147",
    "--check-offline-load",
], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/check_assets.py', '--asset-root', '/content/T2ICount-assets', '--data', 'fsc147', '--check-offline-load'], returncode=0)

## 6. Optional infrastructure smoke tests
Infrastructure smoke tests are already validated. Leave `RUN_INFRA_SMOKE_TESTS=False` for normal experiments. Enable them only after changes to model loading, dependencies, checkpoint/resume, DataLoader/infrastructure, or similar plumbing.

The numerical reference is image `2`, prompt `sea shells`, GT `8`, prediction approximately `8.165655`. When explicitly enabled, detailed combined stdout/stderr and pass/fail diagnostics remain visible.

In [9]:
import datasets

print("datasets:", datasets.__file__)

from datasets.carpk import CARPK
from datasets.dataset import ObjectCount
from models.build import build_t2icount
from utils.paths import AssetPaths

print("Local imports OK")

datasets: /content/T2ICount-Implementation/datasets/__init__.py
Local imports OK


In [10]:
import subprocess
import sys

if not RUN_INFRA_SMOKE_TESTS:
    print("Skipping infrastructure smoke test (RUN_INFRA_SMOKE_TESTS=False)")
else:
    result = subprocess.run(
        [
            sys.executable,
            "test.py",
            "--asset-root", ASSET_ROOT,
            "--data", "fsc147",
            "--batch-size", "1",
            "--max-samples", "1",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(result.stdout)
    print("\nRETURN CODE:", result.returncode)

    if result.returncode != 0:
        print("\nNUMERICAL SMOKE FAILED")
    else:
        print("\nNUMERICAL SMOKE PASSED")

2026-08-16 04:13:39.808504: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-16 04:13:39.888370: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from /content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Removed legacy non-persistent 

## 7. VSCode terminal environment
A VSCode integrated terminal is a separate shell; variables set with `os.environ` in this notebook kernel are not guaranteed to exist there. Run the following in each VSCode/Colab terminal session before launching commands:

```bash
cd /content/T2ICount-Implementation
export T2ICOUNT_ASSET_ROOT="/content/T2ICount-assets"
export HF_HUB_OFFLINE=1
export TRANSFORMERS_OFFLINE=1
```

## 8. Optional mini training smoke run
This validated infrastructure check uses two training samples for one epoch and verifies loading, initialization, forward/loss/backward, optimizer steps, and checkpoint/log persistence to Drive. It runs only when `RUN_INFRA_SMOKE_TESTS=True`; otherwise the cell skips before launching `train.py`. Validation and test splits remain unchanged.

In [11]:
import subprocess
import sys

if not RUN_INFRA_SMOKE_TESTS:
    print("Skipping infrastructure smoke test (RUN_INFRA_SMOKE_TESTS=False)")
else:
    result = subprocess.run(
        [
            sys.executable,
            "train.py",

            "--asset-root", ASSET_ROOT,
            "--save-dir",
            f"{DRIVE_ASSET_ROOT}/checkpoints/baseline_retrain/smoke",
            "--content", "baseline_smoke",
            "--batch-size", "1",
            "--epochs", "1",
            "--smoke-train-samples", "2",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(result.stdout)
    print("\nRETURN CODE:", result.returncode)

    if result.returncode != 0:
        print("\nTRAINING SMOKE FAILED")
    else:
        print("\nTRAINING SMOKE PASSED")

08-16 04:14:11, asset_root:/content/T2ICount-assets
08-16 04:14:11, content:baseline_smoke
08-16 04:14:11, seed:-1
08-16 04:14:11, crop_size:384
08-16 04:14:11, concat_size:224
08-16 04:14:11, downsample_ratio:8
08-16 04:14:11, data_dir:/content/T2ICount-assets/datasets/FSC147
08-16 04:14:11, config:/content/T2ICount-Implementation/configs/v1-inference.yaml
08-16 04:14:11, sd_path:/content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
08-16 04:14:11, clip_path:/content/T2ICount-assets/pretrained/clip-vit-large-patch14
08-16 04:14:11, save_dir:/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/smoke
08-16 04:14:11, max_num:2
08-16 04:14:11, resume:
08-16 04:14:11, batch_size:1
08-16 04:14:11, smoke_train_samples:2
08-16 04:14:11, train_samples:0
08-16 04:14:11, train_subset_seed:3407
08-16 04:14:11, stride:384
08-16 04:14:11, beta:0.0001
08-16 04:14:11, weight_decay:0.0005
08-16 04:14:11, lr:5e-05
08-16 04:14:11, num_workers:0
08-16 04:14:11, start_epoch:0

In [ ]:
from pathlib import Path

CKPT = Path(
    f"{DRIVE_ASSET_ROOT}/checkpoints/baseline_retrain/"
    "smoke/baseline_smoke/0_ckpt.tar"
)

print("Checkpoint:", CKPT)
print("Exists:", CKPT.is_file())

if CKPT.is_file():
    print("Size:", CKPT.stat().st_size / 1024**3, "GB")

Checkpoint: /content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/smoke/baseline_smoke/0_ckpt.tar
Exists: True
Size: 10.739242461510003 GB


### 8.1 Resume training smoke
When `RUN_INFRA_SMOKE_TESTS=True`, verify that the full-state training checkpoint restores model, optimizer, epoch, metrics, and RNG state and continues from the next epoch.

In [13]:
import subprocess
import sys

if not RUN_INFRA_SMOKE_TESTS:
    print("Skipping infrastructure smoke test (RUN_INFRA_SMOKE_TESTS=False)")
else:
    result = subprocess.run(
        [
            sys.executable,
            "train.py",

            "--asset-root", ASSET_ROOT,
            "--save-dir",
            f"{DRIVE_ASSET_ROOT}/checkpoints/baseline_retrain/smoke",
            "--content", "baseline_smoke_resume",
            "--batch-size", "1",
            "--epochs", "2",
            "--smoke-train-samples", "2",
            "--resume", str(CKPT),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(result.stdout)
    print("\nRETURN CODE:", result.returncode)

    if result.returncode != 0:
        print("\nRESUME SMOKE FAILED")
    else:
        print("\nRESUME SMOKE PASSED")

08-16 04:16:44, asset_root:/content/T2ICount-assets
08-16 04:16:45, content:baseline_smoke_resume
08-16 04:16:45, seed:-1
08-16 04:16:45, crop_size:384
08-16 04:16:45, concat_size:224
08-16 04:16:45, downsample_ratio:8
08-16 04:16:45, data_dir:/content/T2ICount-assets/datasets/FSC147
08-16 04:16:45, config:/content/T2ICount-Implementation/configs/v1-inference.yaml
08-16 04:16:45, sd_path:/content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
08-16 04:16:45, clip_path:/content/T2ICount-assets/pretrained/clip-vit-large-patch14
08-16 04:16:45, save_dir:/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/smoke
08-16 04:16:45, max_num:2
08-16 04:16:45, resume:/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/smoke/baseline_smoke/0_ckpt.tar
08-16 04:16:45, batch_size:1
08-16 04:16:45, smoke_train_samples:2
08-16 04:16:45, train_samples:0
08-16 04:16:45, train_subset_seed:3407
08-16 04:16:45, stride:384
08-16 04:16:45, beta:0.0001
08-16 04:16:45

## 9. Limited-compute baseline experiment
The executable pilot below trains a deterministic 500-sample subset for 10 epochs, validates at epoch 9, and streams combined stdout/stderr live. It saves under `checkpoints/baseline_retrain/pilot_500x10/baseline_500x10/`. This is a real experiment and does not use the infrastructure smoke limiter.

The original full-baseline command remains documented here for later use. Resume requires a full-state `.tar` checkpoint, not a weights-only `.pth` file.

```bash
python train.py \
  --asset-root "/content/T2ICount-assets" \
  --save-dir "/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain" \
  --content run_01

# After an interruption, add for example:
# --resume "/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/run_01/50_ckpt.tar"
```

In [ ]:
import subprocess
import sys
import time

PILOT_BATCH_SIZE = 1
TRAIN_SAMPLES = 500
TRAIN_SUBSET_SEED = 3407
EPOCHS = 10

cmd = [
    sys.executable,
    "-u",
    "train.py",

    "--asset-root", ASSET_ROOT,
    "--save-dir",
    f"{DRIVE_ASSET_ROOT}/checkpoints/baseline_retrain/pilot_500x10",

    "--content", "baseline_500x10",

    "--batch-size", str(PILOT_BATCH_SIZE),

    "--train-samples", str(TRAIN_SAMPLES),
    "--train-subset-seed", str(TRAIN_SUBSET_SEED),

    "--epochs", str(EPOCHS),

    "--start-val", "9",
    "--val-epoch", "1",

    "--seed", "3407",
]

start = time.perf_counter()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
elapsed = time.perf_counter() - start

print("\nRETURN CODE:", return_code)
print(f"TOTAL TIME: {elapsed / 60:.1f} min")
if return_code == 0:
    print("LIMITED-COMPUTE BASELINE PASSED")
else:
    print("LIMITED-COMPUTE BASELINE FAILED")